# Threshold-aware query interface over the ECHR table (Layer 3)

Real **SQL over the auditable table** (DuckDB), so Bucket-3 content-aggregate questions are
answered over *columns* instead of generated prose. The key honesty rule:

> **A confidence threshold is field-level abstention.** Any query that touches a low-confidence
> field reports **how many cells were excluded / flagged** at the chosen `min_confidence` —
> low-confidence cells are never silently dropped.

`min_confidence` is a user parameter. An optional **NL→SQL** layer sits behind the existing
`USE_LLM` flag and **degrades gracefully** to direct SQL + canned example queries when no model
is present. The demo is guarded: it prints instructions instead of crashing if the table is
absent.

## 1. Configuration

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR    = Path("../data")
THEMES_FILE = DATA_DIR / "echr_themes.parquet"        # enriched: validated metadata + family-law theme tags
EXTRACT_FILE = DATA_DIR / "echr_extracted.parquet"    # base extraction table (fallback)
TABLE_FILE  = THEMES_FILE if THEMES_FILE.exists() else EXTRACT_FILE
CALIB_FILE  = DATA_DIR / "alienation_calibration.joblib"   # optional (from validation)

USE_LLM   = True                 # False = force direct-SQL only (same pattern as the RAG notebook)
GEN_MODEL    = "llama3.2"           # answer generation (RAG notebook uses the same)
NL2SQL_MODEL = "qwen2.5-coder:3b"   # SQL translation is a coder-model task: llama3.2 (general,
                                     # 3B) measurably copies few-shot answers and drops filters
MIN_CONFIDENCE = 0.70            # default field-level abstention threshold (user parameter)

print(f"table: {TABLE_FILE.name} | USE_LLM={USE_LLM} | default min_confidence={MIN_CONFIDENCE}")

table: echr_themes.parquet | USE_LLM=True | default min_confidence=0.7


## 2. Load the table + register it in DuckDB
The extraction table is loaded into a DataFrame and registered as the DuckDB relation `echr`,
so you can run arbitrary SQL. If a calibration mapping exists, `alienation_conf_eff` is the
**calibrated** confidence; otherwise it falls back to the raw `alienation_conf` (with a note).

In [ ]:
def _load_table():
    if TABLE_FILE.exists():
        return pd.read_parquet(TABLE_FILE)
    csv = TABLE_FILE.with_suffix(".csv")
    return pd.read_csv(csv) if csv.exists() else None


df = _load_table()
con = None
CALIBRATED = False
if df is None:
    print("!! extracted table not found — run echr_extraction.ipynb first.")
else:
    # effective alienation confidence: calibrated if available, else raw
    if "alienation_conf_cal" in df.columns and df["alienation_conf_cal"].notna().any():
        df["alienation_conf_eff"] = df["alienation_conf_cal"].fillna(df["alienation_conf"])
        CALIBRATED = True
    else:
        df["alienation_conf_eff"] = df["alienation_conf"]
    # Judgment year as a column on the SAME table as the theme flags. Without it, a question
    # like "cases per year with primary_theme domestic_violence" had no table to hit: `year`
    # lived only in echr_meta, the themes only here, and NL->SQL failed the binder on
    # whichever half it picked ("does not have a column named year" / "primary_theme not
    # found"). 4/1574 rows carry no judgment_date -> NULL year, reported by _coverage_note.
    df["year"] = pd.to_numeric(df["judgment_date"].astype(str).str[:4],
                               errors="coerce").astype("Int64")
    import duckdb
    con = duckdb.connect()
    con.register("echr", df)
    print(f"registered {len(df)} rows as DuckDB relation 'echr'")
    print("alienation confidence column:",
          "CALIBRATED (alienation_conf_cal)" if CALIBRATED else "RAW (run extraction_validation to calibrate)")
    print("columns:", list(df.columns))


def run_sql(sql):
    if con is None:
        print("no table loaded."); return None
    return con.execute(sql).df()

registered 1116 rows as DuckDB relation 'echr'
alienation confidence column: CALIBRATED (alienation_conf_cal)
columns: ['id', 'title', 'respondent_state', 'articles', 'judgment_date', 'importance', 'genre', 'outcome', 'outcome_conf', 'alienation_alleged', 'alienation_conf', 'alienation_evidence', 'url', 'confidence', 'provenance', 'alienation_conf_cal', 'primary_theme', 'theme_group', 'matched_keywords', 'text_length', 'is_abduction_hague', 'is_adoption', 'is_care_removal', 'is_alienation', 'is_domestic_violence', 'is_custody_residence', 'is_contact_access', 'is_length_procedural', 'alienation_conf_eff']


## 3. Threshold-aware content query (never drops low-confidence silently)
`alienation_at(min_confidence, mode)` answers "which cases allege alienation?" while
**accounting for every low-confidence cell**:
- `mode="filter"` returns only cells at/above the threshold **and reports how many positive
  cells were abstained** (the abstention-zone allegations the rules can't defend);
- `mode="flag"` returns all predicted positives with a `low_confidence` boolean so nothing is
  hidden.

Metadata aggregates carry their own `outcome_conf`; the helper reports it too.

In [ ]:
def alienation_at(min_confidence=MIN_CONFIDENCE, mode="filter"):
    """Cases predicted to allege alienation, threshold-aware. Returns (result_df, audit dict)."""
    if con is None:
        return None, {}
    conf = "alienation_conf_eff"
    base = con.execute(f"""
        SELECT id, title, respondent_state, genre, outcome, alienation_conf, {conf} AS conf_eff,
               alienation_evidence, url
        FROM echr WHERE alienation_alleged = TRUE
    """).df()
    n_pos = len(base)
    passed = base[base.conf_eff >= min_confidence].copy()
    abstained = n_pos - len(passed)
    audit = {"predicted_positive": n_pos, "passed": int(len(passed)),
             "abstained_low_conf": int(abstained), "min_confidence": min_confidence,
             "confidence_basis": "calibrated" if CALIBRATED else "raw"}
    if mode == "flag":
        base["low_confidence"] = base.conf_eff < min_confidence
        return base.sort_values("conf_eff", ascending=False), audit
    return passed.sort_values("conf_eff", ascending=False), audit


def print_audit(audit):
    print(f"  [abstention audit] predicted-positive={audit['predicted_positive']} | "
          f"passed(>= {audit['min_confidence']})={audit['passed']} | "
          f"FLAGGED low-confidence (field-level abstention)={audit['abstained_low_conf']} | "
          f"basis={audit['confidence_basis']}")


print("alienation_at() ready")

alienation_at() ready


## 4. RQ-aligned canned queries (always available — the graceful-degradation floor)
Deterministic, reviewed SQL mapped to specific thesis RQs (Comparative RQ16, Bucket-2/3, Temporal RQ21).
Run any with `run_rq("<key>")` — it prints the **question, how to read the result, and the caveat** next to
the table, so the numbers are self-explaining. These are what you cite; `ask()` (LLM) is exploration only.

In [ ]:
# Canned, RQ-aligned queries — deterministic, reviewed, citable (the graceful-degradation floor).
# Each entry documents the RQ it answers, how to read the result, and its caveat.
# "alienation" (genuine allegation) = alienation_alleged AND alienation_conf_eff >= 0.5.
# THEME columns (primary_theme, theme_group, is_*) come from echr_theme_classify.py — keyword
# topic tags over full text (high recall, approximate precision); distinct from the precise flag.

QUERIES = {
 "rq16_violation_by_state": {
   "rq": "Comparative RQ16",
   "question": "Which respondent States are most often found in VIOLATION when parental alienation is alleged?",
   "howto": "One row per State. alienation_cases = merits cases alleging alienation; violations = how many "
            "ended in a violation; violation_pct = share of them. Read top rows by alienation_cases first.",
   "caveat": "Small N per State; 'violation' may rest on other Art.8 grounds, not alienation specifically.",
   "sql": """
     SELECT respondent_state AS state,
            COUNT(*) AS alienation_cases,
            SUM(CASE WHEN outcome='violation' THEN 1 ELSE 0 END) AS violations,
            ROUND(100.0*AVG(CASE WHEN outcome='violation' THEN 1.0 ELSE 0 END),0) AS violation_pct
     FROM echr
     WHERE genre='merits' AND alienation_alleged AND alienation_conf_eff >= 0.5
     GROUP BY respondent_state
     ORDER BY alienation_cases DESC, violations DESC"""},

 "prevalence_by_genre": {
   "rq": "Bucket-3 headline (prevalence)",
   "question": "How prevalent are genuine alienation allegations across the corpus, by case type?",
   "howto": "pct = % of that genre's cases with a genuine alienation allegation. The minority-class headline.",
   "caveat": "Depends on the validated extractor (see extraction_validation P/R/F1); communicated cases have no outcome.",
   "sql": """
     SELECT genre,
            COUNT(*) AS total_cases,
            SUM(CASE WHEN alienation_alleged AND alienation_conf_eff>=0.5 THEN 1 ELSE 0 END) AS alienation_cases,
            ROUND(100.0*SUM(CASE WHEN alienation_alleged AND alienation_conf_eff>=0.5 THEN 1 ELSE 0 END)/COUNT(*),1) AS pct
     FROM echr GROUP BY genre ORDER BY total_cases DESC"""},

 "cases_per_state": {
   "rq": "Bucket-2 (cases vs population)",
   "question": "How many ECHR cases (and alienation cases) does each respondent State produce?",
   "howto": "total_cases is the denominator you divide by population to get a per-capita litigation rate.",
   "caveat": "Measures propensity to litigate at Strasbourg, NOT prevalence of alienation in that country.",
   "sql": """
     SELECT respondent_state AS state, COUNT(*) AS total_cases,
            SUM(CASE WHEN alienation_alleged AND alienation_conf_eff>=0.5 THEN 1 ELSE 0 END) AS alienation_cases
     FROM echr GROUP BY respondent_state ORDER BY total_cases DESC LIMIT 15"""},

 "alienation_over_time": {
   "rq": "Temporal RQ21",
   "question": "How has the volume of alienation cases evolved over time?",
   "howto": "One row per year: total dated cases vs alienation cases. Look at the trend, not single years.",
   "caveat": "Corpus spans 2000-2025 (extended below 2015 on 2026-07-07); the 4 undated cases form the NULL-year row. The pre-2015 leg is thinner — 607 cases over 15 years vs 963 over 2015-2025 — so read the slope against corpus coverage, not as a filing rate.",
   "sql": """
     SELECT year,
            COUNT(*) AS total_cases,
            SUM(CASE WHEN alienation_alleged AND alienation_conf_eff>=0.5 THEN 1 ELSE 0 END) AS alienation_cases
     FROM echr GROUP BY year ORDER BY year"""},

 "outcome_when_alienation": {
   "rq": "Comparative / outcome link",
   "question": "When alienation is alleged, how do the cases end?",
   "howto": "Outcome distribution across ALL genres of alienation cases (communicated = still pending).",
   "caveat": "Descriptive only (N=35); no causal claim that alleging alienation drives the outcome.",
   "sql": """
     SELECT COALESCE(outcome,'(communicated/pending)') AS outcome, COUNT(*) AS n
     FROM echr WHERE alienation_alleged AND alienation_conf_eff>=0.5
     GROUP BY outcome ORDER BY n DESC"""},

 # ---- THEME layer: classify the WHOLE corpus into family-law subtopics (bigger N) ----
 "theme_distribution": {
   "rq": "Theme overview",
   "question": "What family-law subtopics make up the corpus, and how big is each?",
   "howto": "primary_theme = the single best-fitting subtopic; theme_group = the coarse bucket. pct sums to 100.",
   "caveat": "Keyword topic tags over full text (approximate precision); a case often touches several themes.",
   "sql": """
     SELECT primary_theme, theme_group, COUNT(*) AS cases,
            ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS pct
     FROM echr GROUP BY primary_theme, theme_group ORDER BY cases DESC"""},

 "violation_by_theme": {
   "rq": "Comparative (outcome x theme)",
   "question": "Which family-law subtopics are most likely to end in a violation finding?",
   "howto": "Merits, decided cases only. violation_pct = share of that theme's decided cases found in violation. "
            "Now N is large enough per theme to compare (unlike the alienation-only slice).",
   "caveat": "Theme is a keyword topic tag, not the legal basis of the violation; treat as indicative.",
   "sql": """
     SELECT primary_theme,
            COUNT(*) AS merits_cases,
            SUM(CASE WHEN outcome='violation' THEN 1 ELSE 0 END) AS violations,
            ROUND(100.0*AVG(CASE WHEN outcome='violation' THEN 1.0 ELSE 0 END),0) AS violation_pct
     FROM echr WHERE genre='merits' AND outcome IN ('violation','no violation')
     GROUP BY primary_theme ORDER BY merits_cases DESC"""},

 "theme_group_over_time": {
   "rq": "Temporal (theme mix)",
   "question": "Has the mix of public-protection vs private-dispute cases shifted over time?",
   "howto": "Counts per year by coarse group. Compare the columns' trends, not single years.",
   "caveat": "Corpus spans 2000-2025, but the pre-2015 leg is thinner (607 cases over 15 years vs 963 over 2015-2025) and the newest year lags HUDOC publication; counts depend on publication and keyword matching, not just filing.",
   "sql": """
     SELECT year,
            SUM(CASE WHEN theme_group='public_protection' THEN 1 ELSE 0 END) AS public_protection,
            SUM(CASE WHEN theme_group='private_dispute' THEN 1 ELSE 0 END) AS private_dispute,
            SUM(CASE WHEN theme_group='cross_cutting' THEN 1 ELSE 0 END) AS cross_cutting
     FROM echr GROUP BY year ORDER BY year"""},

 "alienation_within_themes": {
   "rq": "Theme x alienation",
   "question": "Within each subtopic, how many cases carry a GENUINE alienation allegation?",
   "howto": "cases = all cases in the theme; genuine_alienation = those with a real allegation (conf>=0.5). "
            "Shows alienation concentrates in private disputes, not e.g. care/abduction.",
   "caveat": "primary_theme is single-label, so a case counts once; the precise flag is the validated one.",
   "sql": """
     SELECT primary_theme, COUNT(*) AS cases,
            SUM(CASE WHEN alienation_alleged AND alienation_conf_eff>=0.5 THEN 1 ELSE 0 END) AS genuine_alienation
     FROM echr GROUP BY primary_theme ORDER BY genuine_alienation DESC"""},
}

# Back-compat alias so ask()'s fallback menu still works, plus a self-documenting runner.
CANNED = {k: v["sql"] for k, v in QUERIES.items()}


def run_rq(key):
    """Run a canned RQ query and print question + how-to-read + caveat alongside the table."""
    if con is None:
        print("no table loaded."); return None
    q = QUERIES[key]
    print("="*88)
    print(f"[{q['rq']}]  {q['question']}")
    print(f"  How to read : {q['howto']}")
    res = run_sql(q["sql"])
    print(res.to_string(index=False))
    print(f"  ⚠ Caveat    : {q['caveat']}")
    return res


print("RQ queries available via run_rq('<key>'):")
for k, v in QUERIES.items():
    print(f"  - {k:24s} {v['rq']}")

RQ queries available via run_rq('<key>'):
  - rq16_violation_by_state  Comparative RQ16
  - prevalence_by_genre      Bucket-3 headline (prevalence)
  - cases_per_state          Bucket-2 (cases vs population)
  - alienation_over_time     Temporal RQ21
  - outcome_when_alienation  Comparative / outcome link
  - theme_distribution       Theme overview
  - violation_by_theme       Comparative (outcome x theme)
  - theme_group_over_time    Temporal (theme mix)
  - alienation_within_themes Theme x alienation


## 5. Optional NL→SQL via a local coder model (behind `USE_LLM`, degrades gracefully)
Natural-language quantitative questions are translated to SQL by a **coder model**
(`qwen2.5-coder:3b`) with few-shot examples, behind two deterministic refusal nets:
a **deny-list** of known-unextracted content concepts (checked before the model), and
**`EXPLAIN` validation** of every generated statement with one error-carrying retry.

Two measured model-choice findings baked into this design:
- a *general* 3B model (llama3.2) copies few-shot answers verbatim and silently drops
  filters — the coder model translates entities and composed conditions correctly;
- offering the model a `CANNOT_ANSWER` escape makes it **refuse any question requiring
  composition** (the lazy-escape effect) — so refusal is never delegated to the model.

The generated SQL is always printed before its result; the citable path remains the
reviewed canned queries.

In [ ]:
import re
from pathlib import Path

OLLAMA_OK = False
if USE_LLM:
    try:
        import ollama
        ollama.list()
        OLLAMA_OK = True
    except Exception as e:
        print(f"Ollama unavailable ({e}) — NL->SQL disabled, direct SQL + canned queries only")

SCHEMA = ("Table echr(id TEXT, title TEXT, respondent_state TEXT (ISO-3 code: NOR=Norway, "
          "RUS=Russia, POL=Poland, ROU=Romania, UKR=Ukraine, HUN=Hungary, DEU=Germany, "
          "ITA=Italy, HRV=Croatia, BGR=Bulgaria, TUR=Turkey, AUT=Austria, CHE=Switzerland, "
          "FRA=France, GRC=Greece, LTU=Lithuania, MLT=Malta, MDA=Moldova, SVN=Slovenia), "
          "articles TEXT, judgment_date TEXT 'YYYY-MM-DD', "
          "year INT (judgment year, derived from judgment_date -- so themes, outcome and "
          "year are all queryable in THIS one table, no join needed), "
          "importance INT (1=key..4=routine), "
          "genre TEXT[merits|admissibility|communicated], "
          "outcome TEXT[violation|no violation|struck out|inadmissible|admissible|NULL], "
          "outcome_conf DOUBLE, alienation_alleged BOOLEAN, alienation_conf DOUBLE, "
          "alienation_conf_eff DOUBLE (calibrated), alienation_evidence TEXT, url TEXT, "
          "primary_theme TEXT[abduction_hague|alienation|adoption|care_removal|domestic_violence|"
          "custody_residence|contact_access|length_procedural|other], "
          "theme_group TEXT[public_protection|private_dispute|cross_cutting], "
          "is_abduction_hague BOOLEAN, is_alienation BOOLEAN, is_adoption BOOLEAN, "
          "is_care_removal BOOLEAN, is_domestic_violence BOOLEAN, is_custody_residence BOOLEAN, "
          "is_contact_access BOOLEAN, is_length_procedural BOOLEAN). "
          "outcome is NULL for communicated cases. A GENUINE alienation allegation = "
          f"alienation_alleged AND alienation_conf_eff >= {MIN_CONFIDENCE}. "
          "Table echr_meta(id TEXT, respondent TEXT ISO-3, importance INT, year INT, "
          "formation TEXT[GRANDCHAMBER|CHAMBER|COMMITTEE], duration_days INT (proceedings "
          "duration: application introduced -> judgment or final decision), separate_opinion BOOLEAN, "
          "n_scl_citations INT). Table echr_kp(id TEXT, kp_code TEXT) = Registry thesaurus codes.")

# valid-value vocabulary from the registry (auto-refreshes when value_registry.ipynb re-runs)
_vr = Path("../reports/value_registry.json")
if _vr.exists():
    import json as _vrjson
    _reg = _vrjson.loads(_vr.read_text())
    def _vals(table, field, cap=12):
        f = _reg.get(table, {}).get("fields", {}).get(field, {})
        return [k for k in f.get("values", {}) if k != "(other)"][:cap]
    _hints = []
    for tbl, fld in [("echr_themes (derived)", "outcome"), ("echr_themes (derived)", "genre"),
                     ("echr_themes (derived)", "primary_theme"),
                     ("echr_themes (derived)", "theme_group"), ("swiss", "canton")]:
        v = _vals(tbl, fld)
        if v:
            _hints.append(f"{fld} in [{', '.join(map(str, v))}]")
    if _hints:
        SCHEMA = SCHEMA + " Known values: " + "; ".join(_hints) + "."

# few-shot examples: ISO-3 mapping, thresholds, date handling, composed conditions.
# NO refusal example and NO escape instruction: measured with qwen2.5-coder:3b, an offered
# CANNOT_ANSWER escape gets taken whenever the question needs composition (lazy escape) —
# refusals therefore live in the deterministic nets below, never in the model.
FEW_SHOT = [
    ("How many cases against Norway are in the corpus?",
     "SELECT COUNT(*) AS cases FROM echr WHERE respondent_state = 'NOR';"),
    ("What share of merits judgments found a violation?",
     "SELECT ROUND(100.0*AVG(CASE WHEN outcome='violation' THEN 1.0 ELSE 0 END),1) "
     "AS violation_pct FROM echr WHERE genre='merits';"),
    ("Which respondent state has the most confident alienation allegations?",
     f"SELECT respondent_state, COUNT(*) AS n FROM echr WHERE alienation_alleged AND "
     f"alienation_conf_eff >= {MIN_CONFIDENCE} GROUP BY respondent_state ORDER BY n DESC LIMIT 5;"),
    ("Wie viele Entscheidungen stammen aus dem Jahr 2023?",
     "SELECT COUNT(*) AS n FROM echr WHERE year = 2023;"),
    ("What share of merits judgments after 2019 found a violation?",
     "SELECT ROUND(100.0*AVG(CASE WHEN outcome='violation' THEN 1.0 ELSE 0 END),1) "
     "AS violation_pct FROM echr WHERE genre='merits' AND judgment_date > '2019-12-31';"),
    ("How long do proceedings take on average from application to judgment?",
     "SELECT ROUND(AVG(duration_days)/365.25, 1) AS avg_years FROM echr_meta "
     "WHERE duration_days IS NOT NULL;"),
    # "per <column>" = a BREAKDOWN. Measured failure without these: "per primary_theme against
    # Romania" came back as a single COUNT with an invented primary_theme filter, and "against
    # Sweden per primary_theme" grouped by the column it was filtering on.
    ("How many cases are there per primary_theme against Romania?",
     "SELECT primary_theme, COUNT(*) AS n FROM echr WHERE respondent_state = 'ROU' "
     "GROUP BY primary_theme ORDER BY n DESC;"),
    ("How many cases per outcome for merits judgments?",
     "SELECT outcome, COUNT(*) AS n FROM echr WHERE genre = 'merits' "
     "GROUP BY outcome ORDER BY n DESC;"),
    # theme x year: the shape that used to fail. echr carries both columns, so it is one
    # table and no join -- the translator's two wrong answers were echr_meta (no theme)
    # and echr with a `year` column that did not yet exist.
    ("How many cases per year are there with primary_theme domestic_violence?",
     "SELECT year, COUNT(*) AS n FROM echr WHERE primary_theme = 'domestic_violence' "
     "GROUP BY year ORDER BY year;"),
]

# net 4 — GROUPING. A question that asks for a breakdown ("per X", "by X", "je X") must produce
# GROUP BY X. EXPLAIN cannot catch this: the wrong query is perfectly valid SQL, it just answers
# a different question, and the number it returns looks plausible. Deterministic, no model.
_GROUP_PHRASE = re.compile(
    r"\b(?:per|by|for each|je|pro|nach|aufgeschl\u00fcsselt nach)\s+"
    r"([A-Za-z_][A-Za-z_ ]{2,30})", re.IGNORECASE)
_GROUP_SYNONYMS = {                       # phrase -> SQL tokens that satisfy the request
    "theme": ["primary_theme"], "themes": ["primary_theme"],
    "primary theme": ["primary_theme"], "primary_theme": ["primary_theme"],
    "theme group": ["theme_group"], "theme_group": ["theme_group"],
    "state": ["respondent_state"], "country": ["respondent_state"],
    "respondent": ["respondent_state", "respondent"],
    "respondent state": ["respondent_state"], "respondent_state": ["respondent_state"],
    "year": ["year", "judgment_date"], "jahr": ["year", "judgment_date"],
    "outcome": ["outcome"], "genre": ["genre"], "importance": ["importance"],
    "canton": ["canton"], "kanton": ["canton"], "senate": ["senate"], "senat": ["senate"],
    "formation": ["formation"], "article": ["articles"], "articles": ["articles"],
    "court": ["court_type"], "court type": ["court_type"], "dokumenttyp": ["dokumenttyp"],
}


# net 5 — REQUIRED FILTERS. The mirror of net 4: a value the question names out loud must
# appear in the SQL. Measured failure — "how many cases per year with primary_theme
# domestic_violence" produced `SELECT year, COUNT(*) FROM echr_meta GROUP BY year`: the theme
# filter silently vanished and the table chosen has no such column, so the answer was every
# case per year. The registry already knows which values belong to which column, so this is a
# lookup, not a guess.
def _registry_values():
    out = {}
    for entry in (_reg or {}).values():
        for field, meta in entry.get("fields", {}).items():
            if meta.get("kind") != "categorical":
                continue
            for val in meta.get("values", {}):
                if isinstance(val, str) and len(val) > 3 and val != "(other)":
                    out.setdefault(val.lower(), field)
    return out


_REG_VALUES = _registry_values() if _vr.exists() else {}


def required_filters(question):
    """Values the question names that the SQL must therefore filter on."""
    q = (question or "").lower()
    hits = []
    for val, field in _REG_VALUES.items():
        spaced = val.replace("_", " ")
        if val in q or (len(spaced) > 6 and spaced in q):
            hits.append((field, val))
    return hits


def filters_satisfied(sql, required):
    low = (sql or "").lower()
    return [(f, v) for f, v in required
            if v.lower() not in low and v.lower().replace("_", " ") not in low]


def requested_group_by(question):
    """The column a 'per X' phrase asks to break down by, or None. Longest match wins so
    'per primary theme' does not resolve through the bare 'theme' entry."""
    for m in _GROUP_PHRASE.finditer(question or ""):
        tail = re.sub(r"\s+", " ", m.group(1).strip().lower())
        words = tail.split(" ")
        for n in range(min(3, len(words)), 0, -1):
            key = " ".join(words[:n])
            if key in _GROUP_SYNONYMS:
                return key, _GROUP_SYNONYMS[key]
    return None


def group_by_satisfied(sql, tokens):
    m = re.search(r"GROUP\s+BY\s+(.+?)(?:\bORDER\b|\bLIMIT\b|\bHAVING\b|;|$)",
                  sql or "", re.IGNORECASE | re.DOTALL)
    if not m:
        return False
    clause = m.group(1).lower()
    return any(t.lower() in clause for t in tokens)

# net 1 — deny-list of KNOWN-UNEXTRACTED content concepts. This is semantic knowledge about
# what is NOT in the data; no SQL validator (and, measured, no 3B model) can carry it reliably.
UNEXTRACTED_RE = re.compile(
    r"(receiv\w*|erhielt|erh\u00e4lt|awarded|granted|zugesprochen)\s+(the\s+)?(sole\s+)?"
    r"(custody|sorgerecht|obhut)"
    r"|(custody|sorgerecht|obhut)\s+(was\s+)?(receiv|award|grant|zugesprochen)"
    r"|children? liv\w+ with|kind(er)? leb\w+ bei"
    r"|\bmarried\b|\bverheiratet\b",
    re.IGNORECASE)   # proceedings-duration removed 2026-07-07: introductiondate
                     # arrived and duration became Bucket-2 metadata (echr_meta)


def _extract_sql(text):
    """Pull a single SQL statement out of an LLM reply (handles ```sql fences + trailing prose)."""
    m = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S | re.I)  # prefer fenced block
    sql = m.group(1) if m else text
    low = sql.lower()
    starts = [p for p in (low.find("select"), low.find("with")) if p != -1]
    if starts:
        sql = sql[min(starts):]
    sql = sql.split(";")[0].strip()                              # drop anything after first statement
    sql = re.sub(r"<\s*threshold\s*>", str(MIN_CONFIDENCE), sql, flags=re.I)
    return sql + ";" if sql else sql


def validate_sql(sql):
    """net 3 — guardrail: SELECT/WITH-only, then schema-check via EXPLAIN (plans, never
    executes). Returns None if valid, else a short error string."""
    if not sql or sql.startswith("--"):
        return sql or "empty"
    if not re.match(r"\s*(SELECT|WITH)\b", sql, re.I):
        return "not a SELECT statement"
    try:
        con.execute("EXPLAIN " + sql.rstrip(";"))
        return None
    except Exception as e:
        return str(e).splitlines()[0][:200]


def nl2sql(question, hint="", must_filter=None):
    """Deny-list -> few-shot coder-model translation (temp 0) -> EXPLAIN validation with one
    retry carrying the error. Returns executable SQL or a '-- ...' comment on refusal/failure.

    hint / must_filter: what the caller's router already decided about WHICH corpus the
    question is about (see _sql_route in ask.ipynb) -- knowledge the translator cannot derive,
    since the same country name means a national corpus or an ECHR respondent code depending
    on phrasing. `hint` is prose in the prompt; `must_filter` [(column, value), ...] joins
    net 5, so a dropped respondent filter is retried and then refused, never returned as a
    plausible whole-corpus number (measured: "cases against Switzerland per year" came back as
    every case per year, and the prose hint alone did not fix it)."""
    if UNEXTRACTED_RE.search(question or ""):
        return "-- CANNOT_ANSWER: known-unextracted content concept (needs a validated extractor)"
    if not OLLAMA_OK:
        return None
    shots = "\n\n".join(f"Question: {q}\nSQL: {s}" for q, s in FEW_SHOT)
    prompt = (f"You translate questions to a single DuckDB SQL query. Return ONLY SQL, no "
              f"prose. Choose the correct table; adapt the examples to the question actually "
              f"asked.\nSchema: {SCHEMA}\n\n{shots}\n\nQuestion: {question}\nSQL:")
    need_filters = required_filters(question) + list(must_filter or [])
    if need_filters:
        prompt += ("\nThe question names " +
                   " and ".join("%s = '%s'" % f for f in need_filters) +
                   " — the SQL must filter on that, using a table that has the column.\nSQL:")
    if hint:
        prompt += f"\n{hint}\nSQL:"
    want_group = requested_group_by(question)
    if want_group:
        prompt += (f"\nThe question asks for a BREAKDOWN per {want_group[0]}: the SQL must "
                   f"SELECT {want_group[1][0]} alongside the aggregate and GROUP BY "
                   f"{want_group[1][0]}. Do not filter on {want_group[1][0]}.\nSQL:")
    suffix = ""
    err = "generation failed"
    for attempt in range(2):
        try:
            r = ollama.chat(model=NL2SQL_MODEL,
                            messages=[{"role": "user", "content": prompt + suffix}],
                            options={"temperature": 0})
            sql = _extract_sql(r["message"]["content"])
        except Exception as e:
            return f"-- generation failed: {e}"
        err = validate_sql(sql)
        if err is None:
            missing = filters_satisfied(sql, need_filters)
            if missing:
                err = "dropped filter " + ", ".join("%s=%s" % m for m in missing)
                suffix = ("\n\nYour previous SQL (" + sql + ") ignored part of the question. It "
                          "must filter " + " AND ".join("%s = '%s'" % m for m in missing) +
                          ". Choose a table that has those columns.\nSQL:")
                continue
            if want_group and not group_by_satisfied(sql, want_group[1]):
                # valid SQL, wrong question. One targeted retry, then refuse — answering a
                # breakdown question with an ungrouped total is a silent substitution.
                err = f"missing GROUP BY {want_group[1][0]}"
                suffix = (f"\n\nYour previous SQL ({sql}) did not group the result. The "
                          f"question asks for one row PER {want_group[0]}. Return SQL that "
                          f"SELECTs {want_group[1][0]} and GROUP BYs {want_group[1][0]}.\nSQL:")
                continue
            return sql
        suffix = (f"\n\nYour previous SQL was invalid ({err}). "
                  f"Return a corrected single SQL statement only.\nSQL:")
    if err and err.startswith("dropped filter"):
        return ("-- FILTER_DROPPED: the question names " +
                ", ".join("%s=%s" % m for m in filters_satisfied("", need_filters)) +
                " but the translator left it out twice")
    if want_group and err and err.startswith("missing GROUP BY"):
        return (f"-- GROUPING_MISMATCH: the question asks for a breakdown per "
                f"{want_group[0]}, the translator returned an ungrouped query twice")
    return f"-- invalid after retry: {err}"


def ask(question=None):
    if con is None:
        print("no table loaded."); return None
    if question and OLLAMA_OK:
        sql = nl2sql(question)
        print("NL->SQL:", sql)
        if sql and not sql.startswith("--"):
            try:
                return run_sql(sql)
            except Exception as e:
                print("generated SQL failed, falling back to canned menu:", e)
    if question and not OLLAMA_OK:
        print("(no LLM) — canned queries below:")
    print("canned queries: run_rq(<key>) with keys:", ", ".join(QUERIES))
    return None


print("ask() ready — NL->SQL", "ENABLED (coder model, deny-list + EXPLAIN nets)" if OLLAMA_OK else "DISABLED")

ask() ready — NL->SQL ENABLED (coder model, deny-list + EXPLAIN nets)


## 6. Demo — RQ answers (self-documenting) + the honest-abstention method point

In [ ]:
if con is None:
    print("Demo skipped — table not found. Run echr_extraction.ipynb (+ echr_theme_classify.py) first.")
    print(f"   expected: {TABLE_FILE} (or .csv fallback)")
else:
    # (1) THEME layer — the whole corpus classified into family-law subtopics (large N to slice).
    for key in ("theme_distribution", "violation_by_theme", "alienation_within_themes"):
        run_rq(key)
        print()

    # (2) The precise alienation slice + prevalence.
    for key in ("prevalence_by_genre", "rq16_violation_by_state"):
        run_rq(key)
        print()

    # (3) The methodological point: a confidence threshold IS field-level abstention.
    print("="*88)
    print("[Method] HONEST ABSTENTION — same 'which cases allege alienation?' at rising thresholds")
    print("  How to read : raising min_confidence answers fewer cases but abstains (flags, never drops) the rest.")
    for thr in [0.5, 0.7, 0.9]:
        _, a = alienation_at(min_confidence=thr, mode="filter")
        print(f"    min_conf={thr}: {a['passed']:>2} answered | "
              f"{a['abstained_low_conf']:>2} abstained (surfaced, not dropped) | basis={a['confidence_basis']}")

[Theme overview]  What family-law subtopics make up the corpus, and how big is each?
  How to read : primary_theme = the single best-fitting subtopic; theme_group = the coarse bucket. pct sums to 100.
    primary_theme       theme_group  cases  pct
     care_removal public_protection    212 19.0
custody_residence   private_dispute    196 17.6
  abduction_hague   private_dispute    184 16.5
         adoption public_protection    150 13.4
       alienation   private_dispute    124 11.1
   contact_access   private_dispute    101  9.1
            other     cross_cutting     77  6.9
domestic_violence     cross_cutting     62  5.6
length_procedural     cross_cutting     10  0.9
  ⚠ Caveat    : Keyword topic tags over full text (approximate precision); a case often touches several themes.

[Comparative (outcome x theme)]  Which family-law subtopics are most likely to end in a violation finding?
  How to read : Merits, decided cases only. violation_pct = share of that theme's decided cases fou

## 6b. The Bucket-2 surface — every parsed metadata field, registered and queryable

Bucket 2 is only as wide as the metadata actually captured at import. The audit below
**prints per-source field coverage** (that table *defines* the Bucket-2 answerable surface),
then registers one relation per source carrying every quantitatively useful field:

| relation | fields (beyond id/year) | coverage notes |
|---|---|---|
| `echr_meta` | respondent, articles, importance (1\u20134), genre, outcome, separate-opinion flag, represented-by flag, n_scl (Strasbourg case-law citations), n_appnos | scl 27 % (older judgments), the rest \u2265 74 % |
| `ris_meta` | dokumenttyp, senate code (civil `Ob` / criminal `Os`), rechtsgebiete, n_applied (decisions applying a Rechtssatz), normen | normen/rechtsgebiete exist on Rechts\u00e4tze only (36) \u2014 printed, not hidden |
| `ris_normen` | one row per (Rechtssatz, statute) | exploded for GROUP BY |
| `swiss_meta` | canton, court, hierarchy depth, language, abstract flag, text length | 100 % |

Demo queries below deliberately answer questions **only metadata can answer** \u2014 including
one that quantifies a data-curation decision: how much of the RIS keyword harvest is the
*criminal-law* Entfremdung homonym (misappropriation), visible in `rechtsgebiete`/senate.

In [ ]:
import json as _json
import re as _re

SWISS_FILE = DATA_DIR / "swiss_parental_alienation.json"
RIS_FILE   = DATA_DIR / "ris_parental_alienation.json"
ECHR_FILE  = DATA_DIR / "echr_parental_alienation.json"

def _coverage(records, name, skip=("full_text", "content")):
    from collections import Counter
    cov = Counter()
    for r in records:
        for k, v in r.items():
            if k not in skip and v not in (None, "", [], {}):
                cov[k] += 1
    print(f"  {name}: {len(records)} records, {len(cov)} non-empty fields; lowest-coverage 5:",
          {k: f"{c/len(records)*100:.0f}%" for k, c in sorted(cov.items(), key=lambda x: x[1])[:5]})

if con is not None and SWISS_FILE.exists() and RIS_FILE.exists() and ECHR_FILE.exists():
    swiss_raw = _json.loads(SWISS_FILE.read_text())
    ris_raw   = _json.loads(RIS_FILE.read_text())
    echr_raw  = _json.loads(ECHR_FILE.read_text())
    print("metadata coverage audit (the Bucket-2 surface):")
    _coverage(echr_raw, "echr"); _coverage(ris_raw, "ris"); _coverage(swiss_raw, "swiss")

    # --- ECHR: the full quantitative metadata surface (beyond the validated table) ---
    def _n(s, sep=";"):
        return len([x for x in (s or "").split(sep) if x.strip()])
    from datetime import datetime as _dt

    def _pdate(s):
        s = (s or "").split()[0] if s else ""
        try:
            return _dt.strptime(s, "%d/%m/%Y").date()
        except (ValueError, IndexError):
            return None

    def _formation(dc):
        for f in ("GRANDCHAMBER", "CHAMBER", "COMMITTEE"):
            if f in (dc or ""):
                return f
        return None

    def _duration_days(r):
        """Proceedings duration at Strasbourg: application introduced -> judgment or final
        decision (HUDOC populates judgementdate on judgments, decisiondate on admissibility
        decisions; communicated cases are excluded -- still pending). Available since the
        2026-07-07 import added `introductiondate` -- this used to be a Bucket-3 refusal
        example; now it is metadata (coverage ~315/1116, mostly admissibility decisions)."""
        if (r.get("doctype") or "").upper() == "HECOM":
            return None
        a = _pdate(r.get("introductiondate"))
        b = _pdate(r.get("judgementdate")) or _pdate(r.get("decisiondate"))
        return (b - a).days if a and b and b >= a else None

    echr_meta = pd.DataFrame([{
        "id": r.get("itemid"), "respondent": r.get("respondent"),
        "articles": r.get("article"), "importance": int(r["importance"]) if str(r.get("importance", "")).isdigit() else None,
        "doctype": r.get("doctype"),
        "year": int(r["ecli"].split(":")[3][:4]) if _re.match(r"ECLI:CE:ECHR:\d{4}", r.get("ecli") or "") else None,
        "has_violation_finding": bool((r.get("violation") or "").strip()),
        "separate_opinion": str(r.get("separateopinion", "")).upper() == "TRUE",
        "represented": bool((r.get("representedby") or "").strip()),
        "n_scl_citations": _n(r.get("scl")),          # Strasbourg case-law cited by this case
        "n_appnos": _n(r.get("extractedappno")),
        "formation": _formation(r.get("documentcollectionid")),   # GRANDCHAMBER|CHAMBER|COMMITTEE
        "duration_days": _duration_days(r),
    } for r in echr_raw])
    echr_kp = pd.DataFrame([
        {"id": r.get("itemid"), "kp_code": code.strip()}
        for r in echr_raw for code in (r.get("kpthesaurus") or "").split(";") if code.strip()
    ])

    # --- RIS: senate code, legal area, applied-decision counts, statute references ---
    _SEN = _re.compile(r"\d{1,3}\s*([A-Z][a-z]{1,2})")
    def _senate(gz):
        m = _SEN.match((gz or "").split(";")[0].strip())
        return m.group(1) if m else None
    ris_meta = pd.DataFrame([{
        "id": r.get("id"), "dokumenttyp": r.get("dokumenttyp"), "gericht": r.get("gericht"),
        "year": int(r["entscheidungsdatum"][:4]) if (r.get("entscheidungsdatum") or "")[:4].isdigit() else None,
        "senate": _senate(r.get("geschaeftszahl")),
        "rechtsgebiete": r.get("rechtsgebiete") or None,
        "n_applied": int(r.get("n_entscheidungstexte") or 0),   # decisions applying a Rechtssatz
        "rechtssatznummer": r.get("rechtssatznummern") or None,
    } for r in ris_raw])
    ris_normen = pd.DataFrame([
        {"id": r.get("id"), "statute": n.split("\u00a7")[0].strip() or n.strip()}
        for r in ris_raw for n in (r.get("normen") or "").split(";") if n.strip()
    ])

    # --- Swiss: full court-hierarchy surface ---
    swiss_meta = pd.DataFrame([{
        "id": r.get("stable_id"), "canton": r.get("canton"), "court_type": r.get("court_type"),
        "hierarchy_depth": len(r.get("hierarchy") or []),
        "year": r.get("year"), "lang": r.get("lang"),
        "has_abstract": bool((r.get("abstract") or "").strip()),
        "text_length": r.get("text_length"),
    } for r in swiss_raw])

    for name, df_ in [("echr_meta", echr_meta), ("echr_kp", echr_kp), ("ris_meta", ris_meta),
                      ("ris_normen", ris_normen), ("swiss_meta", swiss_meta)]:
        con.register(name, df_)
    print(f"\nregistered: echr_meta ({len(echr_meta)}) | ris_meta ({len(ris_meta)}) | "
          f"ris_normen ({len(ris_normen)}) | swiss_meta ({len(swiss_meta)})\n")

    print("[B2/ECHR] THE MOVED BOUNDARY -- proceedings duration (application -> judgment), by formation:")
    print(run_sql("""
        SELECT doctype, COALESCE(formation, '(untagged)') AS formation, COUNT(*) AS cases,
               ROUND(AVG(duration_days)/365.25, 1) AS avg_years,
               ROUND(MEDIAN(duration_days)/365.25, 1) AS median_years
        FROM echr_meta
        WHERE duration_days IS NOT NULL
        GROUP BY doctype, formation ORDER BY cases DESC""").to_string(index=False))
    print("  How to read : 'how long do proceedings last?' was a Bucket-3 REFUSAL example until the")
    print("                2026-07-07 import added introductiondate -- the bucket boundary is set by")
    print("                data availability, and it moved. Coverage: 315/1116 (introduction date +")
    print("                judgment or decision date; HUDOC tags formation on judgments only).\n")

    print("[B2/ECHR] top kpthesaurus codes (the Registry's own structured legal-concept tags):")
    print(run_sql("""
        SELECT kp_code, COUNT(*) AS cases FROM echr_kp
        GROUP BY kp_code ORDER BY cases DESC LIMIT 8""").to_string(index=False))
    print("  Caveat      : numeric thesaurus codes (labels not exposed via this API); groupable")
    print("                as-is, resolvable by inspection for the top codes.\n")

    print("[B2/ECHR] importance level x separate opinions x citation depth (pure metadata):")
    print(run_sql("""
        SELECT importance, COUNT(*) AS cases,
               ROUND(100.0*AVG(CASE WHEN separate_opinion THEN 1.0 ELSE 0 END),0) AS pct_separate_op,
               ROUND(AVG(n_scl_citations),1) AS avg_cited_cases
        FROM echr_meta WHERE importance IS NOT NULL
        GROUP BY importance ORDER BY importance""").to_string(index=False))
    print("  How to read : importance 1 = key case ... 4 = routine; separate opinions and case-law")
    print("                citation depth track legal salience, computable with zero NLP.\n")

    print("[B2/AT] the Entfremdung homonym, quantified from metadata alone:")
    print(run_sql("""
        SELECT COALESCE(rechtsgebiete, 'decision (no RS metadata)') AS rechtsgebiet,
               senate, COUNT(*) AS records
        FROM ris_meta GROUP BY 1, 2 ORDER BY records DESC LIMIT 8""").to_string(index=False))
    print("  How to read : criminal senates (Os) / Strafrecht = the misappropriation homonym the")
    print("                retrieval corpus excludes -- here the exclusion is a queryable fact.\n")

    print("[B2/AT] most-applied Rechtssaetze (citation weight of each principle):")
    print(run_sql("""
        SELECT rechtssatznummer, n_applied, year
        FROM ris_meta WHERE dokumenttyp='Rechtssatz'
        ORDER BY n_applied DESC LIMIT 5""").to_string(index=False))
    print()

    print("[B2/AT] statutes referenced by the Rechtssaetze (exploded normen):")
    print(run_sql("""
        SELECT statute, COUNT(*) AS refs FROM ris_normen
        GROUP BY statute ORDER BY refs DESC LIMIT 6""").to_string(index=False))
    print("  Caveat      : normen exist on the 36 Rechtssaetze only (coverage printed above).\n")

    print("[B2/CH] Swiss corpus by canton x court level:")
    print(run_sql("""
        SELECT canton, COUNT(*) AS cases, COUNT(DISTINCT court_type) AS courts,
               ROUND(100.0*AVG(CASE WHEN has_abstract THEN 1.0 ELSE 0 END),0) AS pct_abstract
        FROM swiss_meta GROUP BY canton ORDER BY cases DESC LIMIT 8""").to_string(index=False))
    print()

    print("[B2 x-jurisdiction] corpus volume per year per jurisdiction:")
    print(run_sql("""
        SELECT CAST(year AS INT) AS year, SUM(CASE WHEN j='ECHR' THEN n ELSE 0 END) AS echr,
                     SUM(CASE WHEN j='AT'   THEN n ELSE 0 END) AS at_ogh,
                     SUM(CASE WHEN j='CH'   THEN n ELSE 0 END) AS ch
        FROM (
          SELECT year, 'ECHR' AS j, COUNT(*) AS n FROM echr_meta WHERE year IS NOT NULL GROUP BY 1
          UNION ALL SELECT year, 'AT', COUNT(*) FROM ris_meta WHERE year IS NOT NULL GROUP BY 1
          UNION ALL SELECT year, 'CH', COUNT(*) FROM swiss_meta WHERE year IS NOT NULL GROUP BY 1
        ) GROUP BY year HAVING year BETWEEN 2015 AND 2025 ORDER BY year""").to_string(index=False))
    print("\n  Standing caveat: counts describe the keyword-matched corpora, never litigation rates.")
else:
    print("cross-jurisdiction demo skipped (missing raw files or no table).")


metadata coverage audit (the Bucket-2 surface):
  echr: 1116 records, 39 non-empty fields; lowest-coverage 5: {'referencedate': '1%', 'rulesofcourt': '4%', 'publishedby': '5%', 'applicability': '5%', 'externalsources': '12%'}
  ris: 548 records, 30 non-empty fields; lowest-coverage 5: {'schlagworte': '1%', 'veroeffentlicht': '7%', 'geaendert': '7%', 'normen': '7%', 'ecli': '7%'}
  swiss: 2031 records, 23 non-empty fields; lowest-coverage 5: {'abstract': '70%', 'Signatur': '100%', 'Spider': '100%', 'hierarchy': '100%', 'canton': '100%'}

registered: echr_meta (1116) | ris_meta (548) | ris_normen (101) | swiss_meta (2031)

[B2/ECHR] THE MOVED BOUNDARY -- proceedings duration (application -> judgment), by formation:
doctype  formation  cases  avg_years  median_years
  HEDEC (untagged)    307        3.3           2.9
  HEJUD    CHAMBER      8        4.5           3.4
  How to read : 'how long do proceedings last?' was a Bucket-3 REFUSAL example until the
                2026-07-07 import a

## 6c. Closing the loop — the question RAG abstained on, answered here

In `rag_echr_ris.ipynb` the demo query *“Wie viele Urteile betreffen die Durchsetzung
des Kontaktrechts?”* triggers **type-1 abstention**: top-k retrieval cannot count a corpus.
This is where that question is *supposed* to land — the router's refusal is not a dead end
but a **handoff to the query layer**, which answers it deterministically over columns:

1. the **canned SQL answer** (citable), with its caveat printed;
2. the same question through the optional **NL→SQL** layer (local llama3.2) — exploration
   only, shown to demonstrate the degradation path, never cited.

Scope note: the theme layer exists for **ECHR only**, so the deterministic answer is
ECHR-scoped; for AT/CH the same question currently reduces to keyword-metadata counts —
exactly the kind of boundary the bucket typology makes explicit.

In [ ]:
EXAMPLE_ABSTAINED_QUERY = "Wie viele Urteile betreffen die Durchsetzung des Kontaktrechts?"

if con is not None:
    print(f"Q (abstained in RAG, answered here): {EXAMPLE_ABSTAINED_QUERY}\n")

    # (1) deterministic, citable answer over the validated table + theme tags
    print("[1] Canned SQL (citable):")
    df_ans = run_sql("""
        SELECT COUNT(*)                                   AS contact_judgments,
               SUM(CASE WHEN outcome='violation' THEN 1 ELSE 0 END) AS violations
        FROM echr
        WHERE genre = 'merits' AND is_contact_access
    """)
    print(df_ans.to_string(index=False))
    print("  How to read : merits judgments tagged contact_access (keyword theme, >=2 mentions).")
    print("  Caveat      : theme tags are high-recall keyword rules, not a validated classifier;")
    print("                ECHR-scoped -- the AT/CH corpora carry no theme layer (stated boundary).\n")

    # (2) the same question through NL->SQL (exploration only, degrades gracefully)
    print("[2] NL->SQL via local LLM (exploration only, never cited):")
    if OLLAMA_OK:
        sql = nl2sql(EXAMPLE_ABSTAINED_QUERY)
        print("  generated SQL:", (sql or "").replace("\n", " ")[:300])
        try:
            print(run_sql(sql).to_string(index=False))
        except Exception as e:
            print("  generated SQL failed (honest failure, canned path above is the answer):", e)
    else:
        print("  Ollama not reachable -- canned path above is the answer (graceful degradation).")


Q (abstained in RAG, answered here): Wie viele Urteile betreffen die Durchsetzung des Kontaktrechts?

[1] Canned SQL (citable):
 contact_judgments  violations
               425       281.0
  How to read : merits judgments tagged contact_access (keyword theme, >=2 mentions).
  Caveat      : theme tags are high-recall keyword rules, not a validated classifier;
                ECHR-scoped -- the AT/CH corpora carry no theme layer (stated boundary).

[2] NL->SQL via local LLM (exploration only, never cited):
  generated SQL: SELECT COUNT(*) AS n FROM echr WHERE primary_theme = 'contact_access';
  n
101


### 6d. NL→SQL spot-check — two more quantitative questions, end to end
Demonstrates the exploratory path on one EN and one DE question. The generated SQL is
printed **before** its result so it can be reviewed — the citable path remains the canned
queries above. **Observed behaviour is itself a finding:** the 3B local model translates
simple metadata questions correctly but silently drops or invents filters on harder ones
(e.g. ignoring the alienation-allegation condition) — plausible SQL, wrong intent. This is
precisely why NL→SQL sits behind a flag and deterministic reviewed SQL is what the thesis
cites: **for aggregate answers, the trust boundary is the query text, not the number.**

In [ ]:
NL_DEMO_QUESTIONS = [
    "How many merits judgments found a violation of Article 8?",
    "Welcher Staat hat die meisten Faelle mit einer Entfremdungsbehauptung?",
]

if con is not None and OLLAMA_OK:
    for q in NL_DEMO_QUESTIONS:
        print("=" * 88)
        print("Q:", q)
        sql = nl2sql(q)
        print("generated SQL:", (sql or "").replace("\n", " ")[:400])
        try:
            res = run_sql(sql)
            print(res.head(10).to_string(index=False))
        except Exception as e:
            print("generated SQL failed:", e)
elif con is not None:
    print("Ollama not reachable -- NL->SQL demo skipped (canned queries remain available).")


Q: How many merits judgments found a violation of Article 8?
generated SQL: SELECT COUNT(*) AS n FROM echr WHERE genre='merits' AND outcome='violation' AND primary_theme LIKE '%Article 8%';
 n
 0
Q: Welcher Staat hat die meisten Faelle mit einer Entfremdungsbehauptung?
generated SQL: SELECT respondent_state, COUNT(*) AS n FROM echr WHERE alienation_alleged GROUP BY respondent_state ORDER BY n DESC LIMIT 5;
respondent_state  n
             DEU  8
             ROU  8
             BGR  6
             AUT  3
             SVN  3


## 7. How this answers the capability-boundary question
- **The threshold *is* the abstention.** For a content-aggregate question, `min_confidence`
  decides which `alienation_alleged` cells the system is willing to stand behind; every cell
  below it is reported as abstained, not deleted. The user sees the trade-off (more answers vs
  more confidence) explicitly, per query.
- **Metadata vs content are answered differently and honestly.** Outcome-rate questions run
  over high-confidence structured columns; alienation questions run over a calibrated, abstaining
  content column. The same SQL surface serves both, but the confidence accounting makes clear
  which answers are settled metadata and which are model judgements.
- **Reusability.** Because confidence was calibrated **once** offline, the threshold here has a
  stable meaning on unseen ECHR Article 8 cases — the table can grow by re-running extraction
  with no new labels, and this query layer keeps working unchanged.